In [1]:
# === Fantasy PPR Scoring with nflreadpy ===

import numpy as np
import pandas as pd
import nflreadpy as nfl
from typing import Tuple

# Config
ALL_SEASONS = list(range(1999, 2024))  # 1999-2023

def fantasy_week_max(season: int) -> int:
    return 16 if season <= 2020 else 17

def normalize_team(team: str, season: int) -> str:
    t = (team or "").upper()
    if season <= 2015 and t in {"LAR", "LA", "STL"}:
        return "STL"
    if season >= 2016 and t in {"LAR", "LA", "STL"}:
        return "LAR"
    if season <= 2019 and t in {"OAK", "LV"}:
        return "OAK"
    if season >= 2020 and t in {"OAK", "LV"}:
        return "LV"
    if season <= 2016 and t in {"SD", "LAC"}:
        return "SD"
    if season >= 2017 and t in {"SD", "LAC"}:
        return "LAC"
    return t

def base_filter(df: pd.DataFrame, week_min: int, week_max: int) -> pd.DataFrame:
    out = df.query("season_type == 'REG' and @week_min <= week <= @week_max").copy()
    if "play_deleted" in out.columns:
        out = out[out["play_deleted"].fillna(0) != 1]
    if "play_type" in out.columns:
        out = out[out["play_type"] != "no_play"]
    return out

def safe_col(df, name, default=0):
    return df[name].fillna(0) if name in df.columns else default

print("Config loaded successfully")

Config loaded successfully


In [2]:
# Player PPR Scoring
def player_ppr(pbp_reg: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = pbp_reg.copy()
    
    pass_attempt   = (df.get("pass_attempt", df.get("pass", 0)).fillna(0).astype(int) == 1)
    rush_attempt   = (df.get("rush_attempt", df.get("rush", 0)).fillna(0).astype(int) == 1)
    complete_pass  = (df.get("complete_pass", 0).fillna(0).astype(int) == 1)
    
    pass_role = pass_attempt  & (df["fantasy_player_id"] == df["passer_player_id"])
    rush_role = rush_attempt  & (df["fantasy_player_id"] == df["rusher_player_id"])
    recv_role = complete_pass & (df["fantasy_player_id"] == df["receiver_player_id"])
    
    # Calculate fantasy points
    pass_fp = np.where(pass_role,
                       safe_col(df, "passing_yards")*0.04 + safe_col(df, "pass_touchdown")*4 - safe_col(df, "interception")*2,
                       0)
    rush_fp = np.where(rush_role,
                       safe_col(df, "rushing_yards")*0.1 + safe_col(df, "rush_touchdown")*6,
                       0)
    reception_pts = np.where(recv_role, 1, 0)
    recv_yards_td = np.where(recv_role,
                             safe_col(df, "receiving_yards")*0.1 + safe_col(df, "pass_touchdown")*6,
                             0)
    
    # 2-pt conversions
    two_pt_success = (safe_col(df, "two_point_attempt") == 1) & (df.get("two_point_conv_result", "") == "success")
    two_pt_recv = two_pt_success & (safe_col(df, "pass") == 1) & (df["fantasy_player_id"] == df["receiver_player_id"])
    two_pt_rush = two_pt_success & (safe_col(df, "rush") == 1) & (df["fantasy_player_id"] == df["rusher_player_id"])
    two_pt_fp = np.where(two_pt_recv | two_pt_rush, 2, 0)
    
    # Fumbles
    fum_fp = np.where((safe_col(df, "fumble_lost") == 1) & (df["fantasy_player_id"] == df.get("fumbled_1_player_id")),
                      -2, 0)
    
    df["fp"] = pass_fp + rush_fp + reception_pts + recv_yards_td + two_pt_fp + fum_fp
    
    # Season totals
    leaders = (df.groupby(["season","fantasy_player_id","fantasy_player_name"], as_index=False)["fp"].sum()
                 .sort_values(["season","fp"], ascending=[True, False]))
    games = (df.loc[df["fp"] != 0]
               .groupby(["season","fantasy_player_id","fantasy_player_name"], as_index=False)["week"].nunique()
               .rename(columns={"week":"games"}))
    leaders = leaders.merge(games, on=["season","fantasy_player_id","fantasy_player_name"], how="left")
    leaders["ppg"] = leaders["fp"] / leaders["games"].fillna(1)
    
    # Weekly data
    weekly = (df.groupby(["season","fantasy_player_id","fantasy_player_name","week"], as_index=False)["fp"].sum()
                .sort_values(["season","fantasy_player_name","week"]))
    
    return leaders, weekly

print("Fantasy scoring functions loaded")

Fantasy scoring functions loaded


In [3]:
# === Load Roster Positions using nflreadpy (CORRECTED) ===
import nflreadpy as nfl

def load_roster_positions_nflreadpy(seasons: list) -> pd.DataFrame:
    """Load roster data using nflreadpy - CORRECT approach"""
    print(f"Loading roster data for {len(seasons)} seasons using nflreadpy...")
    
    # CORRECT: use 'seasons' parameter, not 'years'
    rosters_pl = nfl.load_rosters(seasons=seasons)  # Returns Polars DataFrame
    rosters = rosters_pl.to_pandas()  # Convert to pandas
    
    print(f"Loaded {len(rosters)} total player-season records")
    
    # Check for required columns
    if 'player_id' not in rosters.columns and 'gsis_id' in rosters.columns:
        rosters['player_id'] = rosters['gsis_id']
    if 'player_name' not in rosters.columns and 'full_name' in rosters.columns:
        rosters['player_name'] = rosters['full_name']
    
    # Standardize positions to fantasy categories
    position_map = {
        'QB': 'QB',
        'RB': 'RB', 'FB': 'RB',
        'WR': 'WR',
        'TE': 'TE',
        'K': 'K'
    }
    
    rosters['position'] = rosters['position'].map(position_map)
    rosters = rosters[rosters['position'].notna()].copy()
    
    # Select only needed columns
    keep_cols = ['season', 'player_id', 'position']
    if 'player_name' in rosters.columns:
        keep_cols.append('player_name')
    
    rosters = rosters[keep_cols].copy()
    
    print(f"Filtered to {len(rosters)} fantasy-relevant player-seasons")
    print("Position breakdown:")
    print(rosters['position'].value_counts())
    
    return rosters

print("Roster loading function ready")

Roster loading function ready


In [4]:
# === Complete Data Processing Pipeline ===

print("="*80)
print("COMPLETE DATA PROCESSING PIPELINE (nflreadpy only)")
print("="*80)

# Step 1: Load roster data
print("\nSTEP 1: Loading roster positions...")
rosters = load_roster_positions_nflreadpy(ALL_SEASONS)

# Step 2: Process PBP data
print("\nSTEP 2: Processing play-by-play data...")
players_all, players_weekly_all = [], []

for yr in ALL_SEASONS:
    print(f"  Processing {yr}...", end=" ")
    
    # Load PBP
    pbp_pl = nfl.load_pbp(seasons=yr)  # nflreadpy uses 'seasons' parameter
    pbp_y = pbp_pl.to_pandas()
    
    # Filter to fantasy weeks
    wk_max = fantasy_week_max(yr)
    pbp_reg = base_filter(pbp_y, week_min=1, week_max=wk_max)
    
    # Memory optimization
    for c in pbp_reg.select_dtypes("float64").columns:
        pbp_reg[c] = pbp_reg[c].astype("float32")
    
    # Calculate fantasy points
    p_season, p_weekly = player_ppr(pbp_reg)
    
    players_all.append(p_season)
    players_weekly_all.append(p_weekly)
    
    print(f"{len(p_season)} players")

# Step 3: Combine and add positions
print("\nSTEP 3: Combining and adding positions...")
players_season = pd.concat(players_all, ignore_index=True)
players_weekly = pd.concat(players_weekly_all, ignore_index=True)

# Merge with position data
players_season = players_season.merge(
    rosters[['season', 'player_id', 'position']],
    left_on=['season', 'fantasy_player_id'],
    right_on=['season', 'player_id'],
    how='left'
).drop(columns=['player_id'])

players_weekly = players_weekly.merge(
    rosters[['season', 'player_id', 'position']],
    left_on=['season', 'fantasy_player_id'],
    right_on=['season', 'player_id'],
    how='left'
).drop(columns=['player_id'])

missing_pos = players_season['position'].isna().sum()
print(f"Players with positions: {len(players_season) - missing_pos}/{len(players_season)}")

print("\n" + "="*80)
print("DATA PROCESSING COMPLETE!")
print("="*80)

# Save outputs
print("\nSaving outputs...")
players_season.to_parquet("players_ppr_with_positions.parquet", index=False)
players_weekly.to_parquet("players_weekly_with_positions.parquet", index=False)
print("✓ Saved!")

COMPLETE DATA PROCESSING PIPELINE (nflreadpy only)

STEP 1: Loading roster positions...
Loading roster data for 25 seasons using nflreadpy...
Loaded 60127 total player-season records
Filtered to 20028 fantasy-relevant player-seasons
Position breakdown:
WR    7057
RB    5327
TE    3838
QB    2788
K     1018
Name: position, dtype: int64

STEP 2: Processing play-by-play data...
  Processing 1999... 541 players
  Processing 2000... 531 players
  Processing 2001... 526 players
  Processing 2002... 600 players
  Processing 2003... 582 players
  Processing 2004... 527 players
  Processing 2005... 513 players
  Processing 2006... 506 players
  Processing 2007... 525 players
  Processing 2008... 520 players
  Processing 2009... 536 players
  Processing 2010... 547 players
  Processing 2011... 564 players
  Processing 2012... 564 players
  Processing 2013... 562 players
  Processing 2014... 565 players
  Processing 2015... 576 players
  Processing 2016... 559 players
  Processing 2017... 558 pla

In [5]:
# ========================================================================
# TEMPORAL SPLITS - CRITICAL for avoiding data leakage
# ========================================================================

TRAIN_SEASONS = list(range(1999, 2020))  # 1999-2019: 21 seasons
VAL_SEASONS   = [2020, 2021]              # 2020-2021: 2 seasons  
TEST_SEASONS  = [2022, 2023]              # 2022-2023: 2 seasons

print("="*80)
print("TEMPORAL SPLIT CONFIGURATION")
print("="*80)
print(f"Train: {TRAIN_SEASONS[0]}-{TRAIN_SEASONS[-1]} ({len(TRAIN_SEASONS)} seasons)")
print(f"Val:   {VAL_SEASONS[0]}-{VAL_SEASONS[-1]} ({len(VAL_SEASONS)} seasons)")
print(f"Test:  {TEST_SEASONS[0]}-{TEST_SEASONS[-1]} ({len(TEST_SEASONS)} seasons)")
print("="*80)

TEMPORAL SPLIT CONFIGURATION
Train: 1999-2019 (21 seasons)
Val:   2020-2021 (2 seasons)
Test:  2022-2023 (2 seasons)


In [6]:
# ========================================================================
# LOAD FEATURES
# ========================================================================

print("Loading features...")
features_train = pd.read_parquet("features_train.parquet")
features_val = pd.read_parquet("features_val.parquet")
features_test = pd.read_parquet("features_test.parquet")

print(f"Train: {len(features_train)} rows")
print(f"Val:   {len(features_val)} rows")
print(f"Test:  {len(features_test)} rows")

# Identify feature columns
exclude_cols = ['season', 'fantasy_player_id', 'fantasy_player_name', 
                'fp', 'ppg', 'games', 'position', 'target_fp', 'target_ppg', 
                'target_top24', 'target_games', 'position_rank', 'fp_above_replacement',
                'target_fp_above_replacement']

feature_cols = [c for c in features_train.columns if c not in exclude_cols]
print(f"\n{len(feature_cols)} features identified")

# Prepare data
X_train = features_train[feature_cols].fillna(0)
y_train = features_train['target_fp'].fillna(0)
pos_train = features_train['position']

X_val = features_val[feature_cols].fillna(0)
y_val = features_val['target_fp'].fillna(0)
pos_val = features_val['position']

X_test = features_test[feature_cols].fillna(0)
y_test = features_test['target_fp'].fillna(0)
pos_test = features_test['position']

print("\nFeatures loaded and prepared")

Loading features...


FileNotFoundError: [Errno 2] No such file or directory: 'features_train.parquet'

In [ ]:

# ========================================================================
# MIXTURE OF EXPERTS WITH MULTIPLE MODEL TYPES
# ========================================================================

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    print("XGBoost not available - install with: pip install xgboost")
    XGBOOST_AVAILABLE = False

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except ImportError:
    print("LightGBM not available - install with: pip install lightgbm")
    LIGHTGBM_AVAILABLE = False

class PositionExpert:
    """
    Position-specific expert with multiple model options
    Tests different model types and selects the best one
    """
    def __init__(self, position: str):
        self.position = position
        self.models = {}
        self.best_model = None
        self.best_model_name = None
        self.best_score = np.inf
        self.feature_importance_ = None
    
    def fit(self, X, y, X_val, y_val):
        """Train multiple models and select the best one"""
        print(f"\\n  Testing models for {self.position}...")
        
        # Model 1: Random Forest
        try:
            rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
            rf.fit(X, y)
            rf_pred = rf.predict(X_val)
            rf_score = mean_absolute_error(y_val, rf_pred)
            self.models['RandomForest'] = {'model': rf, 'score': rf_score}
            print(f"    RandomForest: MAE = {rf_score:.2f}")
            
            if rf_score < self.best_score:
                self.best_score = rf_score
                self.best_model = rf
                self.best_model_name = 'RandomForest'
        except Exception as e:
            print(f"    RandomForest failed: {e}")
        
        # Model 2: Gradient Boosting
        try:
            gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
            gb.fit(X, y)
            gb_pred = gb.predict(X_val)
            gb_score = mean_absolute_error(y_val, gb_pred)
            self.models['GradientBoosting'] = {'model': gb, 'score': gb_score}
            print(f"    GradientBoosting: MAE = {gb_score:.2f}")
            
            if gb_score < self.best_score:
                self.best_score = gb_score
                self.best_model = gb
                self.best_model_name = 'GradientBoosting'
        except Exception as e:
            print(f"    GradientBoosting failed: {e}")
        
        # Model 3: Ridge Regression
        try:
            ridge = Ridge(alpha=1.0, random_state=42)
            ridge.fit(X, y)
            ridge_pred = ridge.predict(X_val)
            ridge_score = mean_absolute_error(y_val, ridge_pred)
            self.models['Ridge'] = {'model': ridge, 'score': ridge_score}
            print(f"    Ridge: MAE = {ridge_score:.2f}")
            
            if ridge_score < self.best_score:
                self.best_score = ridge_score
                self.best_model = ridge
                self.best_model_name = 'Ridge'
        except Exception as e:
            print(f"    Ridge failed: {e}")
        
        # Model 4: XGBoost (if available)
        if XGBOOST_AVAILABLE:
            try:
                xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
                xgb_model.fit(X, y)
                xgb_pred = xgb_model.predict(X_val)
                xgb_score = mean_absolute_error(y_val, xgb_pred)
                self.models['XGBoost'] = {'model': xgb_model, 'score': xgb_score}
                print(f"    XGBoost: MAE = {xgb_score:.2f}")
                
                if xgb_score < self.best_score:
                    self.best_score = xgb_score
                    self.best_model = xgb_model
                    self.best_model_name = 'XGBoost'
            except Exception as e:
                print(f"    XGBoost failed: {e}")
        
        # Model 5: LightGBM (if available)
        if LIGHTGBM_AVAILABLE:
            try:
                lgb_model = lgb.LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
                lgb_model.fit(X, y)
                lgb_pred = lgb_model.predict(X_val)
                lgb_score = mean_absolute_error(y_val, lgb_pred)
                self.models['LightGBM'] = {'model': lgb_model, 'score': lgb_score}
                print(f"    LightGBM: MAE = {lgb_score:.2f}")
                
                if lgb_score < self.best_score:
                    self.best_score = lgb_score
                    self.best_model = lgb_model
                    self.best_model_name = 'LightGBM'
            except Exception as e:
                print(f"    LightGBM failed: {e}")
        
        print(f"  Best model for {self.position}: {self.best_model_name} (MAE={self.best_score:.2f})")
        
        # Store feature importance from best model
        if hasattr(self.best_model, 'feature_importances_'):
            self.feature_importance_ = self.best_model.feature_importances_
        
        return self
    
    def predict(self, X):
        """Predict using the best model"""
        if self.best_model is None:
            return np.zeros(len(X))
        return self.best_model.predict(X)
    
    def get_model_comparison(self) -> Dict:
        """Return comparison of all models"""
        return self.models

class MoEFantasyOptimizer:
    """
    Complete MoE Framework for Fantasy Football
    Tests multiple model types per position and selects the best
    """
    def __init__(self):
        self.experts = {}
        self.model_comparisons = {}
        self.feature_names = None
    
    def create_experts(self, positions: List[str]):
        """Initialize expert models for each position"""
        for pos in positions:
            self.experts[pos] = PositionExpert(position=pos)
        print(f"\\nCreated {len(self.experts)} expert models: {list(self.experts.keys())}")
    
    def train_experts(self, X_train, y_train, pos_train, X_val, y_val, pos_val):
        """Train each expert on its position-specific data"""
        print("\\n" + "="*80)
        print("TRAINING EXPERTS WITH MULTIPLE MODEL TYPES")
        print("="*80)
        
        for pos, expert in self.experts.items():
            # Filter to this position
            train_mask = pos_train == pos
            val_mask = pos_val == pos
            
            X_pos_train = X_train[train_mask]
            y_pos_train = y_train[train_mask]
            X_pos_val = X_val[val_mask]
            y_pos_val = y_val[val_mask]
            
            if len(X_pos_train) > 0 and len(X_pos_val) > 0:
                expert.fit(X_pos_train, y_pos_train, X_pos_val, y_pos_val)
                
                # Store model comparison
                self.model_comparisons[pos] = expert.get_model_comparison()
            else:
                print(f"\\n  {pos}: Insufficient data (train={len(X_pos_train)}, val={len(X_pos_val)})")
    
    def predict(self, X, positions):
        """Predict using appropriate expert for each position"""
        predictions = np.zeros(len(X))
        
        for pos in self.experts.keys():
            pos_mask = positions == pos
            if pos_mask.sum() > 0:
                predictions[pos_mask] = self.experts[pos].predict(X[pos_mask])
        
        return predictions
    
    def evaluate(self, X_test, y_test, pos_test):
        """Evaluate on test set"""
        print("\\n" + "="*80)
        print("EVALUATION ON TEST SET")
        print("="*80)
        
        y_pred = self.predict(X_test, pos_test)
        
        # Overall metrics
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        print(f"\\nOverall Test Performance:")
        print(f"  MAE:  {mae:.2f} fantasy points")
        print(f"  RMSE: {rmse:.2f} fantasy points")
        print(f"  RÂ²:   {r2:.4f}")
        
        # Per-position metrics
        print(f"\\nPer-Position Performance:")
        for pos in self.experts.keys():
            pos_mask = pos_test == pos
            if pos_mask.sum() > 0:
                pos_mae = mean_absolute_error(y_test[pos_mask], y_pred[pos_mask])
                pos_r2 = r2_score(y_test[pos_mask], y_pred[pos_mask])
                print(f"  {pos}: MAE={pos_mae:.2f}, RÂ²={pos_r2:.4f}")
        
        return {'mae': mae, 'rmse': rmse, 'r2': r2, 'predictions': y_pred}

print("MoE Framework with Multiple Model Types loaded")


MoE Framework with Multiple Model Types loaded


In [ ]:

# ========================================================================
# TRAIN AND EVALUATE MoE MODEL
# ========================================================================

print("\\n" + "="*80)
print("STARTING MoE TRAINING")
print("="*80)

# Create and train MoE
moe = MoEFantasyOptimizer()
moe.create_experts(['QB', 'RB', 'WR', 'TE', 'K'])
moe.train_experts(X_train, y_train, pos_train, X_val, y_val, pos_val)

# Evaluate on test set
test_results = moe.evaluate(X_test, y_test, pos_test)

print("\\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)


\n================================================================================
STARTING MoE TRAINING
\nCreated 5 expert models: ['QB', 'RB', 'WR', 'TE', 'K']
\n================================================================================
TRAINING EXPERTS WITH MULTIPLE MODEL TYPES
\n  Testing models for QB...
    RandomForest: MAE = 17.70
    GradientBoosting: MAE = 18.56
    Ridge: MAE = 18.43
    XGBoost: MAE = 18.80
    LightGBM: MAE = 20.36
  Best model for QB: RandomForest (MAE=17.70)
\n  Testing models for RB...
    RandomForest: MAE = 51.48
    GradientBoosting: MAE = 49.87
    Ridge: MAE = 51.76
    XGBoost: MAE = 56.22
    LightGBM: MAE = 53.12
  Best model for RB: GradientBoosting (MAE=49.87)
\n  Testing models for WR...
    RandomForest: MAE = 52.16
    GradientBoosting: MAE = 50.18
    Ridge: MAE = 50.39
    XGBoost: MAE = 54.72
    LightGBM: MAE = 52.46
  Best model for WR: GradientBoosting (MAE=50.18)
\n  Testing models for TE...
    RandomForest: MAE = 34.06
    Gr

In [ ]:

# ========================================================================
# MODEL COMPARISON SUMMARY
# ========================================================================

print("\\n" + "="*80)
print("MODEL COMPARISON BY POSITION")
print("="*80)

for pos, models in moe.model_comparisons.items():
    print(f"\\n{pos}:")
    for model_name, model_info in sorted(models.items(), key=lambda x: x[1]['score']):
        print(f"  {model_name:20s}: MAE = {model_info['score']:.2f}")
    print(f"  {'â†’ BEST':20s}: {moe.experts[pos].best_model_name} (MAE = {moe.experts[pos].best_score:.2f})")

print("\\n" + "="*80)


\n================================================================================
MODEL COMPARISON BY POSITION
\nQB:
  RandomForest        : MAE = 17.70
  Ridge               : MAE = 18.43
  GradientBoosting    : MAE = 18.56
  XGBoost             : MAE = 18.80
  LightGBM            : MAE = 20.36
  â†’ BEST            : RandomForest (MAE = 17.70)
\nRB:
  GradientBoosting    : MAE = 49.87
  RandomForest        : MAE = 51.48
  Ridge               : MAE = 51.76
  LightGBM            : MAE = 53.12
  XGBoost             : MAE = 56.22
  â†’ BEST            : GradientBoosting (MAE = 49.87)
\nWR:
  GradientBoosting    : MAE = 50.18
  Ridge               : MAE = 50.39
  RandomForest        : MAE = 52.16
  LightGBM            : MAE = 52.46
  XGBoost             : MAE = 54.72
  â†’ BEST            : GradientBoosting (MAE = 50.18)
\nTE:
  GradientBoosting    : MAE = 32.06
  Ridge               : MAE = 32.15
  RandomForest        : MAE = 34.06
  XGBoost             : MAE = 35.62
  LightGBM         